Importing the libraries.

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


from pyspark.sql.types import StructType, StructField, StringType
from pyspark.dbutils import DBUtils

In [ ]:
# Databricks notebook source


# COMMAND ----------

catalog_name = "streaming1"
db_name = "silver"
table_name='shopify_orders'



dbutils.widgets.dropdown("trigger_available_now", "False", ["True", "False"])
trigger_available_now = dbutils.widgets.get("trigger_available_now") == "True"

notebook_name = DBUtils(spark).notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get().split("/")[-1].split(".")[0]
checkpoint_path = f"/Volumes/{catalog_name}/{db_name}/checkpoints/{notebook_name}/"
print(checkpoint_path)



/Volumes/streaming1/silver/checkpoints/(silver FV) Real-time Data Processing with Azure Databricks (and Event Hubs)-2/


#### Silver Layer

In [ ]:
from pyspark.sql.functions import col, from_json, explode
from pyspark.sql.types import ArrayType




from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.dbutils import DBUtils

# Databricks notebook source


# COMMAND ----------

catalog_name = "streaming1"
db_name = "bronze"
db_name1 = "silver"
table_name='shopify_orders'
eventHubName1 = "streamingeventhubs"
key_vault='testScope1'
connector='testsecrettyler'



dbutils.widgets.dropdown("trigger_available_now", "False", ["True", "False"])
trigger_available_now = dbutils.widgets.get("trigger_available_now") == "True"

notebook_name = DBUtils(spark).notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get().split("/")[-1].split(".")[0]
checkpoint_path = f"/Volumes/{catalog_name}/{db_name}/checkpoints/{notebook_name}/"




try:
    spark.sql(f"create catalog {catalog_name} managed location 'abfss://streamingdata-demo@dataengineerdemoweather.dfs.core.windows.net/';")
except:
    print('check if catalog already exists')
try:
    spark.sql(f"create schema if not exists {catalog_name}.{db_name} ;") 
except:
    print('check if bronze schema already exists')

try:
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.{db_name}.checkpoints;") 
except:
    print('check if bronze checkpoints already exists')




# Config
# Replace with your Event Hub namespace, name, and key
connectionString = dbutils.secrets.get(key_vault,connector)
eventHubName =eventHubName1


ehConf = {
  'eventhubs.connectionString' : sc._jvm.org.apache.spark.eventhubs.EventHubsUtils.encrypt(connectionString),
  'eventhubs.eventHubName': eventHubName
}

# Reading stream: Load data from Azure Event Hub into DataFrame 'df' using the previously configured settings
df = spark.readStream \
    .format("eventhubs") \
    .options(**ehConf) \
    .load() \

# Displaying stream: Show the incoming streaming data for visualization and debugging purposes
df.display()

# Writing stream: Persist the streaming data to a Delta table 'streaming.bronze.weather' in 'append' mode with checkpointing
df.writeStream\
    .option("checkpointLocation", checkpoint_path)\
    .outputMode("append")\
    .format("delta")\
    .toTable(f"{catalog_name}.{db_name}.{table_name}")

# Databricks notebook source


# COMMAND ----------


dbutils.widgets.dropdown("trigger_available_now", "False", ["True", "False"])
trigger_available_now = dbutils.widgets.get("trigger_available_now") == "True"

notebook_name = DBUtils(spark).notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get().split("/")[-1].split(".")[0]
checkpoint_path = f"/Volumes/{catalog_name}/{db_name1}/checkpoints/{notebook_name}/"
print(checkpoint_path)



try:
    spark.sql(f"create schema if not exists {catalog_name}.{db_name1} ;") 
except:
    print('check if silver schema already exists')

try:
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.{db_name1}.checkpoints;") 
except:
    print('check if silver checkpoints already exists')





def process_streaming_data(spark, catalog_name, db_name, table_name, checkpoint_path, json_schema, explode_field=None):
    """
    Processes streaming data from a Delta table, applies a JSON schema, and writes it to another Delta table.

    Parameters:
        spark (SparkSession): The Spark session.
        catalog_name (str): The name of the catalog.
        db_name (str): The name of the database.
        table_name (str): The name of the Delta table.
        checkpoint_path (str): Path for checkpointing.
        json_schema (StructType): The JSON schema to apply to the incoming data.
        explode_field (str, optional): Field to explode if it's an array. Default is None.
    """
    # Reading and transforming data
    df = spark.readStream\
        .format("delta")\
        .table(f"{catalog_name}.bronze.{table_name}")\
        .withColumn("body", col("body").cast("string"))\
        .withColumn("body", from_json(col("body"), json_schema))
    
    # Selecting fields
    selected_columns = [col(f"body.{field.name}") for field in json_schema.fields]
    
    if explode_field:
        df = df.select(*selected_columns, col("enqueuedTime").alias('timestamp'), explode(col(f"body.{explode_field}")).alias("Input_array"))
        df = df.select(*selected_columns, 'timestamp', 
                       *[col(f'Input_array.{field.name}').alias(field.name) for field in explode_field.schema.fields])
    else:
        df = df.select(*selected_columns, col("enqueuedTime").alias('timestamp'))

    # Displaying stream
    df.display()

    # Writing stream
    df.writeStream\
        .option("checkpointLocation", checkpoint_path)\
        .outputMode("append")\
        .format("delta")\
        .toTable(f"{catalog_name}.{db_name}.{table_name}")

# Example usage for both schemas
weather_json_schema = StructType([
    StructField("temperature", StringType(), True),
    StructField("time", StringType(), True),
    StructField("skycondition", StringType(), True)
])

cancel_json_schema = StructType([
    StructField("id", StringType(), True),
    StructField("cancel_reason", StringType(), True),
    StructField("cancelled_at", StringType(), True),
    StructField("checkout_id", StringType(), True),
    StructField("created_at", StringType(), True),
    StructField("customer_locale", StringType(), True),
    StructField("financial_status", StringType(), True),
    StructField("presentment_currency", StringType(), True),
    StructField("processed_at", StringType(), True),
    StructField("subtotal_price", StringType(), True),
    StructField("billing_address", StructType([
        StructField('province', StringType()),
        StructField('country', StringType())
    ]), True),
    StructField("line_items", ArrayType(StructType([
        StructField('product_id', StringType()),
        StructField('fulfillable_quantity', StringType()),
        StructField('price', StringType())
    ])), True)
])

# Call the function for weather data
process_streaming_data(spark, catalog_name, db_name, "weather_table", checkpoint_path, weather_json_schema)

# Call the function for cancellation data
process_streaming_data(spark, catalog_name, db_name, "cancellation_table", checkpoint_path, cancel_json_schema, explode_field="line_items")


id,cancel_reason,cancelled_at,checkout_id,created_at,customer_locale,financial_status,presentment_currency,processed_at,subtotal_price,province,country,product_id,fulfillable_quantity,price
6145856831786,null,null,37682609324330,2024-09-08T21:41:38-04:00,en-CA,paid,CAD,2024-09-08T21:41:37-04:00,240.00,Quebec,Canada,9433793495338,1,70.00
6145856831786,null,null,37682609324330,2024-09-08T21:41:38-04:00,en-CA,paid,CAD,2024-09-08T21:41:37-04:00,240.00,Quebec,Canada,9433768526122,1,170.00
6145862598954,null,null,37682624921898,2024-09-08T21:50:07-04:00,en-CA,paid,CAD,2024-09-08T21:50:06-04:00,360.00,Quebec,Canada,9433791234346,1,140.00
6145862598954,null,null,37682624921898,2024-09-08T21:50:07-04:00,en-CA,paid,CAD,2024-09-08T21:50:06-04:00,360.00,Quebec,Canada,9433762726186,1,220.00
6145863745834,null,null,37682627805482,2024-09-08T21:51:47-04:00,en-CA,paid,CAD,2024-09-08T21:51:46-04:00,339.90,Quebec,Canada,9433758531882,1,179.95
6145863745834,null,null,37682627805482,2024-09-08T21:51:47-04:00,en-CA,paid,CAD,2024-09-08T21:51:46-04:00,339.90,Quebec,Canada,9433760694570,1,159.95
6145554776362,null,null,37682049941802,2024-09-08T15:49:17-04:00,en-CA,paid,CAD,2024-09-08T15:49:16-04:00,360.00,Quebec,Canada,9433762726186,1,220.00
6145554776362,null,null,37682049941802,2024-09-08T15:49:17-04:00,en-CA,paid,CAD,2024-09-08T15:49:16-04:00,360.00,Quebec,Canada,9433791234346,1,140.00
6145597210922,null,null,37682120786218,2024-09-08T16:24:53-04:00,en-CA,paid,CAD,2024-09-08T16:24:52-04:00,170.00,Quebec,Canada,9433768526122,1,170.00
